# Pore Segmentation via Hough Transform — Author: Dev KUMAR - PA3

The goal of this work is to use the Hough transform to locate the pores visible in metal-foam CT scan slices, then extract quantitative information from them.

The notebook is organized to directly answer the four points of the assignment:
1. tuning the detection parameters;
2. number of detected circles and associated statistics;
3. reducing over-segmentation;
4. estimating porosity from the retained circles.

## 1. Loading the data

We start by automatically locating the `input` folder, then load the `.tif` volumes.
As in the starting notebook, the analysis is performed on the central axial slice of each volume.

In [ ]:
%matplotlib inline
import math
import os
from glob import glob

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from skimage import color
from skimage.draw import circle_perimeter, disk
from skimage.feature import canny
from skimage.io import imread
from skimage.transform import hough_circle, hough_circle_peaks

# Simple search for the folder containing the scans
CANDIDATE_DIRS = [
    os.path.join('..', 'input'),
]

base_dir = None
for d in CANDIDATE_DIRS:
    if len(glob(os.path.join(d, '*.tif'))) > 0:
        base_dir = d
        break

if base_dir is None:
    raise FileNotFoundError("No input folder containing .tif files was found.")

print('Folder used:', base_dir)

## 2. Overview of the central slices

This step provides a quick check that the data is read correctly and that the porous structures are visible across the different images.

In [ ]:
scan_paths = sorted(glob(os.path.join(base_dir, '*.tif')))
print(len(scan_paths), 'scans found')

In [ ]:
cols = 3
rows = math.ceil(len(scan_paths) / cols)
fig, axes = plt.subplots(rows, cols, figsize=(18, 5 * rows))
axes = np.atleast_1d(axes).reshape(rows, cols)

for ax in axes.flatten():
    ax.axis('off')

for i, (c_path, ax) in enumerate(zip(scan_paths, axes.flatten())):
    c_img = imread(c_path)
    middle_slice = c_img[c_img.shape[0] // 2]
    ax.imshow(middle_slice, cmap='gray')
    ax.set_title(f'{i:02d} - {os.path.basename(c_path)}')

plt.tight_layout()
plt.show()

## 3. Tuning the detection parameters

In the notebook provided as a starting point, detection relied on a fairly unconstrained Hough transform, which generated many redundant circles over the same area.

To get a cleaner result, the following settings were used:
- slightly stronger smoothing before edge detection;
- a narrower radius range;
- a more reasonable maximum number of peaks;
- a minimum distance between detected centers;
- a threshold on the Hough accumulator to eliminate responses that are too weak.

The goal is not to obtain a perfect detection, but a reasonable compromise between the number of detected pores and the readability of the result.

In [ ]:
# Pixel size given in the assignment
PIXEL_SIZE_MM = 0.19

# Settings used for detection
SIGMA = 2.5
RADIUS_MIN = 8
RADIUS_MAX = 70
RADIUS_STEP = 2
TOTAL_NUM_PEAKS = 120
MIN_XDISTANCE = 12
MIN_YDISTANCE = 12
THRESHOLD_REL = 0.35

# Additional filtering to avoid an accumulation of small false positives
SMALL_RADIUS_LIMIT = 16
SMALL_CIRCLE_FACTOR = 1.35

# Settings used for duplicate filtering
DUPLICATE_DIST = 12
DUPLICATE_RADIUS = 6

## 4. Processing functions

The functions below handle:
- slice normalization;
- circle detection;
- duplicate removal;
- statistics computation;
- building a binary mask for porosity estimation.

In [ ]:
def normalize_slice(img_2d):
    """Converts a 2D slice to 8-bit to make processing uniform."""
    img_2d = img_2d.astype(np.float32)
    if img_2d.max() == 0:
        return np.zeros_like(img_2d, dtype=np.uint8)
    return np.clip(255 * (img_2d / img_2d.max()), 0, 255).astype(np.uint8)


def remove_duplicate_circles(cx, cy, radii, accums, dist_thresh=12, radius_thresh=6):
    """
    Simple over-segmentation reduction:
    - sort circles by decreasing confidence;
    - keep the most reliable circle;
    - remove circles that are too close with a similar radius.
    """
    if len(radii) == 0:
        return np.array([]), np.array([]), np.array([]), np.array([])

    order = np.argsort(accums)[::-1]
    cx = np.asarray(cx)[order]
    cy = np.asarray(cy)[order]
    radii = np.asarray(radii)[order]
    accums = np.asarray(accums)[order]

    removed = np.zeros(len(radii), dtype=bool)
    keep = []

    for i in range(len(radii)):
        if removed[i]:
            continue
        keep.append(i)

        for j in range(i + 1, len(radii)):
            if removed[j]:
                continue

            center_dist = np.sqrt((cx[i] - cx[j])**2 + (cy[i] - cy[j])**2)
            radius_diff = abs(radii[i] - radii[j])

            if center_dist < dist_thresh and radius_diff < radius_thresh:
                removed[j] = True

    keep = np.array(keep, dtype=int)
    return cx[keep], cy[keep], radii[keep], accums[keep]


def detect_circles_data(in_img):
    """
    Circle detection improved relative to the base notebook.
    The function now returns the useful data rather than just a displayable image.
    """
    edges = canny(in_img, sigma=SIGMA)
    hough_radii = np.arange(RADIUS_MIN, RADIUS_MAX, RADIUS_STEP)
    hough_res = hough_circle(edges, hough_radii)

    if hough_res.size == 0 or np.max(hough_res) == 0:
        overlay = np.zeros((in_img.shape[0], in_img.shape[1], 3), dtype=np.uint8)
        return {
            'overlay': overlay,
            'edges': edges,
            'cx': np.array([]),
            'cy': np.array([]),
            'radii': np.array([]),
            'accums': np.array([]),
        }

    base_threshold_abs = THRESHOLD_REL * np.max(hough_res)

    accums, cx, cy, radii = hough_circle_peaks(
        hough_res,
        hough_radii,
        total_num_peaks=TOTAL_NUM_PEAKS,
        min_xdistance=MIN_XDISTANCE,
        min_ydistance=MIN_YDISTANCE,
        threshold=base_threshold_abs,
    )

    # Stricter filtering for small circles
    if len(radii) > 0:
        small_threshold_abs = SMALL_CIRCLE_FACTOR * base_threshold_abs
        keep_small = (radii >= SMALL_RADIUS_LIMIT) | (
            (radii < SMALL_RADIUS_LIMIT) & (accums >= small_threshold_abs)
        )
        accums = accums[keep_small]
        cx = cx[keep_small]
        cy = cy[keep_small]
        radii = radii[keep_small]

    # Removing redundant circles: the main answer to question 3
    cx, cy, radii, accums = remove_duplicate_circles(
        cx, cy, radii, accums,
        dist_thresh=DUPLICATE_DIST,
        radius_thresh=DUPLICATE_RADIUS,
    )

    overlay = np.zeros((in_img.shape[0], in_img.shape[1], 3), dtype=np.uint8)
    colors = plt.cm.nipy_spectral(np.linspace(0, 1, max(len(radii), 1)))

    for center_y, center_x, radius, (r, g, b, _) in zip(cy, cx, radii, colors):
        rr, cc = circle_perimeter(center_y, center_x, radius, shape=in_img.shape)
        overlay[rr, cc] = (int(255 * r), int(255 * g), int(255 * b))

    return {
        'overlay': overlay,
        'edges': edges,
        'cx': np.asarray(cx),
        'cy': np.asarray(cy),
        'radii': np.asarray(radii),
        'accums': np.asarray(accums),
    }


def build_pore_mask(img_shape, cx, cy, radii):
    """Builds a binary pore mask from the retained circles."""
    mask = np.zeros(img_shape, dtype=bool)
    for x, y, r in zip(cx, cy, radii):
        rr, cc = disk((y, x), r, shape=img_shape)
        mask[rr, cc] = True
    return mask


def compute_stats(result, img_shape, filename, image_index):
    """Statistics requested in question 2 + porosity for question 4."""
    radii = result['radii']
    cx = result['cx']
    cy = result['cy']

    mask = build_pore_mask(img_shape, cx, cy, radii)
    porosity_percent = 100.0 * mask.sum() / mask.size

    if len(radii) == 0:
        return {
            'image_index': image_index,
            'filename': filename,
            'n_circles': 0,
            'r_min_px': np.nan,
            'r_max_px': np.nan,
            'r_mean_px': np.nan,
            'r_std_px': np.nan,
            'd_min_mm': np.nan,
            'd_max_mm': np.nan,
            'd_mean_mm': np.nan,
            'd_std_mm': np.nan,
            'x_mean_px': np.nan,
            'y_mean_px': np.nan,
            'porosity_percent': porosity_percent,
        }

    diameters_px = 2 * radii
    diameters_mm = diameters_px * PIXEL_SIZE_MM

    return {
        'image_index': image_index,
        'filename': filename,
        'n_circles': int(len(radii)),
        'r_min_px': float(np.min(radii)),
        'r_max_px': float(np.max(radii)),
        'r_mean_px': float(np.mean(radii)),
        'r_std_px': float(np.std(radii)),
        'd_min_mm': float(np.min(diameters_mm)),
        'd_max_mm': float(np.max(diameters_mm)),
        'd_mean_mm': float(np.mean(diameters_mm)),
        'd_std_mm': float(np.std(diameters_mm)),
        'x_mean_px': float(np.mean(cx)),
        'y_mean_px': float(np.mean(cy)),
        'porosity_percent': float(porosity_percent),
    }

## 5. Processing the different images

Detection is now applied to all central slices, and the results are gathered image by image into a table.

In [ ]:
results = []
all_stats = []

for i, c_path in enumerate(scan_paths):
    volume = imread(c_path)
    middle_slice = volume[volume.shape[0] // 2]
    img_u8 = normalize_slice(middle_slice)

    result = detect_circles_data(img_u8)
    stats = compute_stats(result, img_u8.shape, os.path.basename(c_path), i)

    results.append(result)
    all_stats.append(stats)

df_stats = pd.DataFrame(all_stats)
df_stats

## 6. Results for question 2

The table below gathers, for each image:
- the number of retained circles;
- the minimum, maximum, mean radius and standard deviation;
- the diameters converted to millimeters;
- the estimated surface porosity of the slice.

These values answer the quantitative part of question 2.

In [ ]:
df_stats_display = df_stats[[
    'image_index', 'filename', 'n_circles',
    'r_min_px', 'r_max_px', 'r_mean_px', 'r_std_px',
    'd_min_mm', 'd_max_mm', 'd_mean_mm', 'd_std_mm',
    'porosity_percent'
]].copy()

df_stats_display

In [ ]:
# Image chosen to illustrate question 2
img_id = 5 if len(results) > 5 else 0

cx = results[img_id]['cx']
cy = results[img_id]['cy']
radii = results[img_id]['radii']
diameters_mm = 2 * radii * PIXEL_SIZE_MM if len(radii) > 0 else np.array([])

print(f"IMAGE {img_id:02d}: {df_stats.iloc[img_id]['filename']}")
print('> Number of circles:', int(df_stats.iloc[img_id]['n_circles']))
print('> Max radius:', round(df_stats.iloc[img_id]['r_max_px'], 2), '| Min:', round(df_stats.iloc[img_id]['r_min_px'], 2))
print('> Mean radius:', round(df_stats.iloc[img_id]['r_mean_px'], 2), '| Std dev:', round(df_stats.iloc[img_id]['r_std_px'], 2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Distribution of detected centers
axes[0].scatter(cx, cy, c=radii if len(radii) else None, cmap='viridis', s=30, edgecolors='k')
axes[0].invert_yaxis()
axes[0].set_title(f'Center distribution - Image {img_id:02d}')
axes[0].set_xlabel('x coordinate (cx)')
axes[0].set_ylabel('y coordinate (cy)')
if len(radii):
    sc = axes[0].collections[0]
    plt.colorbar(sc, ax=axes[0], label='Radius (px)')

# Diameter histogram
axes[1].hist(diameters_mm, bins=12, edgecolor='black')
axes[1].set_title(f'Diameter distribution - Image {img_id:02d}')
axes[1].set_xlabel('Diameter (mm)')
axes[1].set_ylabel('Number of circles')

plt.tight_layout()
plt.show()

### Analysis of the chosen example

In the image studied below, the distribution of centers shows that the detected pores are present across most of the slice, though not perfectly uniformly.
The histogram also highlights a majority of small and medium diameters, with only a few large pores.

This visualization complements the numeric table above: we now have both statistical and spatial information.

## 7. Answer to question 3: reducing over-segmentation

The main problem observed initially is the presence of several circles very close to each other over the same cavity.
To limit this effect, a duplicate-removal step was added:

1. circles are sorted by decreasing confidence score;
2. the most reliable circle is kept first;
3. circles too close to it, with a similar radius, are removed;
4. the operation is repeated until the end of the list.

As a result, a given area is generally represented by a single circle, making detection more consistent.

## 8. Answer to question 4: porosity estimation

To estimate porosity, a binary mask is built from the retained circles.
Each detected pore fills a region of the mask, and the surface porosity is obtained by computing the ratio:

$$
\text{Porosity} = 100 \times \frac{\text{pore area}}{\text{total image area}}
$$

This method is more reliable than a simple sum of areas, since it avoids counting the same area twice when two circles partially overlap.

In [ ]:
mean_porosity = df_stats['porosity_percent'].mean()
print(f"Estimated mean porosity of the sample: {mean_porosity:.2f} %")

In [ ]:
cols = 3
rows = math.ceil(len(scan_paths) / cols)
fig, axes = plt.subplots(rows, cols, figsize=(18, 5 * rows))
axes = np.atleast_1d(axes).reshape(rows, cols)

for ax in axes.flatten():
    ax.axis('off')

for i, (c_path, ax) in enumerate(zip(scan_paths, axes.flatten())):
    volume = imread(c_path)
    middle_slice = volume[volume.shape[0] // 2]
    img_u8 = normalize_slice(middle_slice)

    overlay = results[i]['overlay']
    stack_img = np.concatenate([plt.cm.bone(img_u8 / 255.0)[:, :, :3], overlay / 255.0], axis=1)

    ax.imshow(stack_img)
    ax.set_title(
        f"{i:02d} - {os.path.basename(c_path)}\n"
        f"Porosity: {df_stats.iloc[i]['porosity_percent']:.1f}% | N={int(df_stats.iloc[i]['n_circles'])}"
    )

plt.tight_layout()
plt.show()

### Reading the summary figure

The figure above allows a quick comparison across the different slices.
Some images contain very few detected pores, while others show a noticeably larger porous surface. The average of the measured porosities across all slices then gives an overall estimate for the sample.

## 9. Saving the results

The results can be exported to `.csv` for easy reuse in the report.

In [ ]:
save_results = True

if save_results:
    df_stats.to_csv('results_td4_per_image.csv', index=False)
    print('File created: results_td4_per_image.csv')